# Basic Machine Learning Model - Philipp

In [69]:
# Relevant imports

from datetime import datetime
import pandas as pd
import sys
from pathlib import Path
import numpy as np
import re
from time import perf_counter
import yaml
import matplotlib.pyplot as plt
from __future__ import annotations


from xgboost import XGBClassifier
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, roc_auc_score



repo_root = Path.cwd().parent.parent
notebook_dir = Path.cwd().parent

for p in [repo_root, notebook_dir]:
    if str(p) not in sys.path:
        sys.path.append(str(p))

from sintef_project.kaggle_metric import score
from sintef_project.submission import create_submission

from data_loaders import Training_set

## Helper Functions for Data Handling

In [70]:
"""Long-format models without autoregressive commitment lags."""

TARGET_PATTERN = re.compile(r"^result_committed_(?P<unit>.+)_t(?P<hour>\d+)$")
T = 168
JOINT_GROUP_SEPARATOR = "::"


def wide_to_long_X(X_wide: pd.DataFrame) -> pd.DataFrame:
    """Convert one wide row per run into one row per run and horizon hour."""
    hourly_bases = sorted({
        column.rsplit("_t", 1)[0]
        for column in X_wide.columns
        if "_t" in column
        and column.rsplit("_t", 1)[1].isdigit()
        and not column.startswith(("price_hour_sin_", "price_hour_cos_"))
    })
    daily_bases = sorted({
        column.rsplit("_day", 1)[0]
        for column in X_wide.columns
        if "_day" in column and column.rsplit("_day", 1)[1].isdigit()
    })
    static_columns = [
        column for column in X_wide.columns
        if column not in {"Run No", "starttime"}
        and "_t" not in column
        and "_day" not in column
    ]

    hour_of_day = np.tile(np.arange(T, dtype=np.int16) % 24, len(X_wide))
    columns = [
        np.sin(2 * np.pi * hour_of_day / 24),
        np.cos(2 * np.pi * hour_of_day / 24),
    ]
    for base in hourly_bases:
        columns.append(
            X_wide[[f"{base}_t{hour}" for hour in range(T)]]
            .to_numpy(dtype=np.float32)
            .ravel()
        )
    for base in daily_bases:
        daily_values = X_wide[
            [f"{base}_day{day}" for day in range(7)]
        ].to_numpy(dtype=np.float32)
        columns.append(np.repeat(daily_values, 24, axis=1).ravel())
    for column in static_columns:
        columns.append(np.repeat(X_wide[column].to_numpy(dtype=np.float32), T))

    column_names = ["hour_sin", "hour_cos"] + hourly_bases + daily_bases + static_columns
    return pd.DataFrame(np.column_stack(columns), columns=column_names)


def get_unit_names(y_columns: pd.Index) -> list[str]:
    """Extract unique generator names from wide target columns."""
    units = {
        match.group("unit")
        for column in y_columns
        if (match := TARGET_PATTERN.match(column))
    }
    if not units:
        raise ValueError("No commitment target columns were found.")
    return sorted(units)


def get_joint_generator_groups(generator_locations):
    """Return generator groups that share one physical location."""
    return [list(generators) for generators in generator_locations.values() if len(generators) > 1]


def _positive_class_probability(model, features: pd.DataFrame) -> np.ndarray:
    probabilities = model.predict_proba(features)
    if len(model.classes_) == 1:
        return probabilities[:, 0] if model.classes_[0] == 1 else probabilities[:, 0] * 0
    return probabilities[:, 1]


def _stage1_count_probability_matrix(model, X_long: pd.DataFrame, n_rows: int, n_units: int) -> np.ndarray:
    """Return Stage-1 hourly count probabilities with shape (n_rows, T, n_units + 1)."""
    class_probabilities = model.predict_proba(X_long)
    class_matrix = np.zeros((len(X_long), n_units + 1), dtype=np.float32)
    count_values = getattr(model, "_joint_count_values", np.arange(n_units + 1))
    class_matrix[:, count_values[model.classes_.astype(int)]] = class_probabilities
    return class_matrix.reshape(n_rows, T, n_units + 1)


def _resolve_stage2_extra_features(X_wide: pd.DataFrame, stage2_extra_inputs=None) -> pd.DataFrame | None:
    """Extract optional weekly features from X_wide for the Stage-2 combination classifiers."""
    if stage2_extra_inputs is None:
        return None
    if callable(stage2_extra_inputs):
        extra_df = stage2_extra_inputs(X_wide)
        if not isinstance(extra_df, pd.DataFrame):
            extra_df = pd.DataFrame(extra_df, index=X_wide.index)
        return extra_df.reset_index(drop=True).astype(np.float32)

    requested = list(stage2_extra_inputs)
    if not requested:
        return None

    excluded = {"Run No", "starttime"}
    selected_columns = [
        col for col in X_wide.columns
        if col not in excluded
        and any(col == pattern or col.startswith(pattern) for pattern in requested)
    ]
    if not selected_columns:
        raise ValueError(
            f"None of the requested stage2_extra_inputs matched columns in X_wide: {requested}"
        )
    return X_wide[selected_columns].reset_index(drop=True).astype(np.float32)


def build_stage2_weekly_features(
    X_wide: pd.DataFrame,
    count_prob_matrix: np.ndarray,
    stage2_extra_inputs=None,
) -> pd.DataFrame:
    """Build one row per week for Stage-2 generator-subset classification.

    By default, uses only the Stage-1 count forecast across the 168 horizon hours.
    Additional weekly inputs from X_wide can be passed via `stage2_extra_inputs`
    (a list of column names/prefixes or a callable taking X_wide).
    """
    n_rows, horizon, n_count_classes = count_prob_matrix.shape
    counts = np.arange(n_count_classes, dtype=np.float32)
    expected_counts = np.tensordot(count_prob_matrix, counts, axes=([2], [0]))  # (n_rows, T)

    feature_blocks = [expected_counts]
    column_names = [f"stage1_expected_count_t{hour}" for hour in range(horizon)]

    for k in range(n_count_classes):
        prob_k = count_prob_matrix[:, :, k]
        feature_blocks.append(prob_k)
        column_names.extend([f"stage1_prob_count_{k}_t{hour}" for hour in range(horizon)])

    summary_blocks = [
        expected_counts.mean(axis=1, keepdims=True),
        expected_counts.std(axis=1, keepdims=True),
        expected_counts.max(axis=1, keepdims=True),
    ]
    summary_names = [
        "stage1_expected_count_mean",
        "stage1_expected_count_std",
        "stage1_expected_count_max",
    ]
    for k in range(n_count_classes):
        prob_k = count_prob_matrix[:, :, k]
        summary_blocks.extend([
            prob_k.mean(axis=1, keepdims=True),
            prob_k.max(axis=1, keepdims=True),
        ])
        summary_names.extend([
            f"stage1_prob_count_{k}_mean",
            f"stage1_prob_count_{k}_max",
        ])

    weekly_array = np.hstack(feature_blocks + summary_blocks).astype(np.float32, copy=False)
    weekly_df = pd.DataFrame(weekly_array, columns=column_names + summary_names)

    extra_df = _resolve_stage2_extra_features(X_wide, stage2_extra_inputs)
    if extra_df is not None:
        weekly_df = pd.concat([weekly_df, extra_df], axis=1)

    return weekly_df


def _bitmask_to_unit_vector(bitmask: int, n_units: int) -> np.ndarray:
    return np.array([(bitmask >> idx) & 1 for idx in range(n_units)], dtype=np.float32)


def _bitmask_to_label(bitmask: int, group: list[str]) -> str:
    return "+".join(unit for idx, unit in enumerate(group) if (bitmask >> idx) & 1)


def _fit_stage2_weekly_combination_models(
    stage2_estimator,
    X_stage2_weekly: pd.DataFrame,
    target_3d: np.ndarray,
    group: list[str],
) -> dict[int, object]:
    """Fit one weekly classifier per active generator count k in {1, ..., M - 1}.

    For each week where `k` generators run simultaneously at least once, the target
    class is the dominant (most frequent) combination of `k` generators in that week.
    """
    n_weeks, _, n_units = target_3d.shape
    hourly_counts = target_3d.sum(axis=2)
    bit_weights = (1 << np.arange(n_units, dtype=np.uint16)).reshape(1, 1, n_units)
    hourly_bitmasks = (target_3d.astype(np.uint16) * bit_weights).sum(axis=2)

    stage2_models = {}
    for k in range(1, n_units):
        valid_weeks = []
        weekly_dominant_masks = []
        weekly_weights = []

        for week_idx in range(n_weeks):
            mask_k = hourly_counts[week_idx] == k
            hours_with_k = int(mask_k.sum())
            if hours_with_k == 0:
                continue
            masks_in_week, counts_in_week = np.unique(
                hourly_bitmasks[week_idx, mask_k],
                return_counts=True,
            )
            dominant_mask = int(masks_in_week[np.argmax(counts_in_week)])
            valid_weeks.append(week_idx)
            weekly_dominant_masks.append(dominant_mask)
            weekly_weights.append(hours_with_k)

        if not valid_weeks:
            continue

        observed_masks = np.unique(weekly_dominant_masks)
        mask_to_class = {int(mask): idx for idx, mask in enumerate(observed_masks)}
        y_stage2 = np.array([mask_to_class[m] for m in weekly_dominant_masks], dtype=np.uint8)
        X_stage2_k = X_stage2_weekly.iloc[valid_weeks].reset_index(drop=True)
        sample_weight = np.array(weekly_weights, dtype=np.float32)

        if len(observed_masks) == 1:
            clf = DummyClassifier(strategy="constant", constant=0)
            clf.fit(X_stage2_k, y_stage2)
        else:
            clf = clone(stage2_estimator)
            if len(observed_masks) == 2:
                clf.set_params(objective="binary:logistic", eval_metric="logloss", num_class=None)
            else:
                clf.set_params(
                    objective="multi:softprob",
                    eval_metric="mlogloss",
                    num_class=len(observed_masks),
                )
            clf.fit(X_stage2_k, y_stage2, sample_weight=sample_weight)

        clf._combination_bitmasks = observed_masks
        clf._combination_unit_matrix = np.vstack([
            _bitmask_to_unit_vector(int(mask), n_units)
            for mask in observed_masks
        ])
        clf._combination_labels = [
            _bitmask_to_label(int(mask), group)
            for mask in observed_masks
        ]
        stage2_models[k] = clf

    return stage2_models


def _predict_stage2_unit_probabilities_given_k(
    stage2_model,
    X_stage2_weekly: pd.DataFrame,
    n_units: int,
    k: int,
    use_proba: bool = True,
) -> np.ndarray:
    """Return weekly P(unit_i is on | k generators are on) with shape (n_weeks, n_units)."""
    n_weeks = len(X_stage2_weekly)
    if stage2_model is None:
        return np.full((n_weeks, n_units), k / n_units, dtype=np.float32)

    comb_unit_matrix = stage2_model._combination_unit_matrix  # (n_combs, n_units)
    n_combs = len(comb_unit_matrix)
    if n_combs == 1:
        return np.repeat(comb_unit_matrix, n_weeks, axis=0)

    raw_proba = stage2_model.predict_proba(X_stage2_weekly)
    aligned_proba = np.zeros((n_weeks, n_combs), dtype=np.float32)
    aligned_proba[:, stage2_model.classes_.astype(int)] = raw_proba

    if not use_proba:
        chosen_comb = aligned_proba.argmax(axis=1)
        return comb_unit_matrix[chosen_comb]

    return aligned_proba @ comb_unit_matrix


def fit_long_models_with_progress(
    estimator,
    X_train_wide,
    y_train_wide,
    joint_groups=None,
    stage2_estimator=None,
    stage2_extra_inputs=None,
    stage2_use_proba: bool = True,
):
    """Fit independent models and two-stage joint generator models.

    For multi-generator groups (`joint_groups`), training proceeds in two stages:
      1. Stage 1 (hourly): Predicts how many generators in the plant run at each hour.
      2. Stage 2 (weekly): For each count k in {1, ..., M - 1}, a trained tree predicts
         centrally for the entire week which k generators are active, based on the
         168-hour Stage-1 forecast and optional `stage2_extra_inputs` from X_train_wide.
    """
    X_long = wide_to_long_X(X_train_wide)
    n_train_rows = len(X_train_wide)
    units = get_unit_names(y_train_wide.columns)
    joint_groups = joint_groups or []
    grouped_units = {unit for group in joint_groups for unit in group}
    unknown_units = grouped_units.difference(units)
    if unknown_units:
        raise ValueError(f"Joint-model generators are missing from targets: {sorted(unknown_units)}")
    models = {}
    training_start = perf_counter()
    training_groups = list(joint_groups)
    training_groups.extend([[unit] for unit in units if unit not in grouped_units])
    base_stage2_estimator = stage2_estimator if stage2_estimator is not None else estimator

    for unit_number, group in enumerate(training_groups, start=1):
        target_start = perf_counter()
        target_columns = {
            unit: [f"result_committed_{unit}_t{hour}" for hour in range(T)]
            for unit in group
        }
        missing = [column for columns in target_columns.values() for column in columns if column not in y_train_wide]
        if missing:
            raise ValueError(f"Missing target columns for {group}: {missing[:3]}")

        X_unit = X_long
        if len(group) == 1:
            model_key = group[0]
            target_values = y_train_wide[target_columns[model_key]].to_numpy(dtype=np.uint8).ravel()
            joint_class_codes = None
            target_3d = None
        else:
            model_key = JOINT_GROUP_SEPARATOR.join(group)
            target_3d = np.stack([
                y_train_wide[target_columns[unit]].to_numpy(dtype=np.uint8)
                for unit in group
            ], axis=2)  # (n_train_rows, T, n_units)
            target_values = target_3d.sum(axis=2).ravel()
            joint_class_codes = np.unique(target_values)
            target_values = pd.Series(target_values).map(
                {code: index for index, code in enumerate(joint_class_codes)}
            ).to_numpy(dtype=np.uint8)

        if np.unique(target_values).size == 1:
            model = DummyClassifier(strategy="constant", constant=target_values[0])
        else:
            model = clone(estimator)
            if len(group) > 1:
                model.set_params(
                    objective="multi:softprob",
                    eval_metric="mlogloss",
                    num_class=len(joint_class_codes),
                )
        model.fit(X_unit, target_values)

        if len(group) > 1:
            model._joint_count_values = joint_class_codes
            count_prob_train = _stage1_count_probability_matrix(
                model,
                X_unit,
                n_rows=n_train_rows,
                n_units=len(group),
            )
            X_stage2_train = build_stage2_weekly_features(
                X_train_wide,
                count_prob_train,
                stage2_extra_inputs=stage2_extra_inputs,
            )
            model._stage2_models = _fit_stage2_weekly_combination_models(
                stage2_estimator=base_stage2_estimator,
                X_stage2_weekly=X_stage2_train,
                target_3d=target_3d,
                group=group,
            )
            model._stage2_extra_inputs = stage2_extra_inputs
            model._stage2_use_proba = stage2_use_proba
        models[model_key] = model

        elapsed = (perf_counter() - training_start) / 60
        last_model = (perf_counter() - target_start) / 60
        eta = (elapsed / unit_number) * (len(training_groups) - unit_number)
        stage2_info = ""
        if len(group) > 1:
            combs_per_k = {
                k: len(clf._combination_labels)
                for k, clf in getattr(model, "_stage2_models", {}).items()
            }
            stage2_info = f" (Stage-2 weekly trees per count k: {combs_per_k})"
        print(
            f"[{unit_number}/{len(training_groups)}] {model_key}{stage2_info} finished | "
            f"last model: {last_model:.2f} min | elapsed: {elapsed:.2f} min | "
            f"ETA: {eta:.2f} min",
            flush=True,
        )

    return models


def positive_class_probabilities(fitted_models, X_wide, y_columns):
    """Predict each generator and reconstruct the original wide target shape."""
    X_long = wide_to_long_X(X_wide)
    n_rows = len(X_wide)
    predictions = {}

    for model_key, model in fitted_models.items():
        if JOINT_GROUP_SEPARATOR not in model_key:
            probabilities = _positive_class_probability(model, X_long)
            probability_matrix = probabilities.reshape(n_rows, T)
            units = [model_key]
            unit_probabilities = probability_matrix[:, :, np.newaxis]
        else:
            units = model_key.split(JOINT_GROUP_SEPARATOR)
            n_units = len(units)
            count_prob_matrix = _stage1_count_probability_matrix(
                model,
                X_long,
                n_rows=n_rows,
                n_units=n_units,
            )  # (n_rows, T, n_units + 1)
            X_stage2_weekly = build_stage2_weekly_features(
                X_wide,
                count_prob_matrix,
                stage2_extra_inputs=getattr(model, "_stage2_extra_inputs", None),
            )
            stage2_models = getattr(model, "_stage2_models", {})
            use_proba = getattr(model, "_stage2_use_proba", True)

            # Combine Stage-1 hourly count probabilities with Stage-2 weekly subset assignment:
            # P(unit_i = 1 at hour t) = P(count_t = M) + sum_{k=1}^{M-1} P(count_t = k) * P_week(unit_i = 1 | count = k)
            unit_probabilities = np.repeat(
                count_prob_matrix[:, :, n_units : n_units + 1],
                n_units,
                axis=2,
            ).astype(np.float32)
            for k in range(1, n_units):
                unit_prob_given_k = _predict_stage2_unit_probabilities_given_k(
                    stage2_models.get(k),
                    X_stage2_weekly,
                    n_units=n_units,
                    k=k,
                    use_proba=use_proba,
                )  # (n_rows, n_units)
                unit_probabilities += (
                    count_prob_matrix[:, :, k : k + 1] * unit_prob_given_k[:, np.newaxis, :]
                )

        for unit_index, unit in enumerate(units):
            for hour in range(T):
                predictions[f"result_committed_{unit}_t{hour}"] = unit_probabilities[:, hour, unit_index]

    result = pd.DataFrame(predictions, index=X_wide.index)
    missing = [column for column in y_columns if column not in result]
    if missing:
        raise ValueError(f"Missing prediction columns: {missing[:3]}")
    return result.loc[:, list(y_columns)]


def evaluate_generator_performance(y_true, predictions):
    """Return 2022 performance metrics for every generator."""
    if not y_true.index.equals(predictions.index):
        raise ValueError("y_true and predictions must have identical indexes.")

    rows = []
    for unit in get_unit_names(y_true.columns):
        target_columns = [f"result_committed_{unit}_t{hour}" for hour in range(T)]
        missing_true = [column for column in target_columns if column not in y_true]
        missing_predictions = [column for column in target_columns if column not in predictions]
        if missing_true or missing_predictions:
            raise ValueError(f"Missing target columns for {unit}.")

        actual = y_true[target_columns].to_numpy(dtype=np.uint8).ravel()
        predicted = predictions[target_columns].to_numpy(dtype=float).ravel()
        predicted_labels = (predicted >= 0.5).astype(np.uint8)
        roc_auc = (
            roc_auc_score(actual, predicted)
            if np.unique(actual).size == 2
            else np.nan
        )
        rows.append({
            "generator": unit,
            "roc_auc": roc_auc,
            "accuracy_at_0.5": accuracy_score(actual, predicted_labels),
            "actual_commitment_rate": actual.mean(),
            "predicted_commitment_rate_at_0.5": predicted_labels.mean(),
            "observations": len(actual),
        })

    return pd.DataFrame(rows).sort_values("roc_auc", ascending=False, na_position="last").reset_index(drop=True)


def _build_true_count_probability_matrix(target_3d: np.ndarray, n_units: int) -> np.ndarray:
    """Build one-hot ground-truth Stage-1 count matrix of shape (n_weeks, T, n_units + 1)."""
    n_weeks, horizon, _ = target_3d.shape
    true_counts = target_3d.sum(axis=2)  # (n_weeks, T)
    true_prob_matrix = np.zeros((n_weeks, horizon, n_units + 1), dtype=np.float32)
    week_idx = np.arange(n_weeks)[:, None]
    hour_idx = np.arange(horizon)[None, :]
    true_prob_matrix[week_idx, hour_idx, true_counts] = 1.0
    return true_prob_matrix


def _predict_stage2_combination_classes(stage2_model, X_stage2_weekly: pd.DataFrame, n_combs: int) -> tuple[np.ndarray, np.ndarray]:
    """Return aligned combination probabilities (n_weeks, n_combs) and predicted class indices."""
    n_weeks = len(X_stage2_weekly)
    if n_combs == 1:
        aligned_proba = np.ones((n_weeks, 1), dtype=np.float32)
        return aligned_proba, np.zeros(n_weeks, dtype=int)
    raw_proba = stage2_model.predict_proba(X_stage2_weekly)
    aligned_proba = np.zeros((n_weeks, n_combs), dtype=np.float32)
    aligned_proba[:, stage2_model.classes_.astype(int)] = raw_proba
    return aligned_proba, aligned_proba.argmax(axis=1)


def evaluate_two_stage_details(fitted_models, X_wide: pd.DataFrame, y_true: pd.DataFrame, block_hours: int = 24) -> dict[str, pd.DataFrame]:
    """Evaluate Stage 1 (count prediction) and Stage 2 (with predicted vs. true inputs)."""
    X_long = wide_to_long_X(X_wide)
    n_rows = len(X_wide)

    stage1_summary_rows = []
    stage1_per_count_rows = []
    stage1_by_24h_rows = []
    stage2_comparison_rows = []
    stage2_ablation_rows = []

    for model_key, model in fitted_models.items():
        if JOINT_GROUP_SEPARATOR not in model_key:
            continue

        units = model_key.split(JOINT_GROUP_SEPARATOR)
        n_units = len(units)
        target_3d = np.stack([
            y_true[[f"result_committed_{unit}_t{hour}" for hour in range(T)]].to_numpy(dtype=np.uint8)
            for unit in units
        ], axis=2)  # (n_rows, T, n_units)
        true_counts = target_3d.sum(axis=2)  # (n_rows, T)

        # Stage 1 predictions
        count_prob_pred = _stage1_count_probability_matrix(model, X_long, n_rows=n_rows, n_units=n_units)
        pred_counts = count_prob_pred.argmax(axis=2)  # (n_rows, T)
        expected_counts = np.tensordot(count_prob_pred, np.arange(n_units + 1, dtype=np.float32), axes=([2], [0]))

        flat_true_counts = true_counts.ravel()
        flat_pred_counts = pred_counts.ravel()
        flat_expected_counts = expected_counts.ravel()
        flat_count_probs = count_prob_pred.reshape(-1, n_units + 1)

        present_classes = [k for k in range(n_units + 1) if np.any(flat_true_counts == k)]
        per_k_aucs = []
        for k in range(n_units + 1):
            true_is_k = (flat_true_counts == k).astype(np.uint8)
            pred_is_k = (flat_pred_counts == k).astype(np.uint8)
            prob_k = flat_count_probs[:, k]
            k_auc = roc_auc_score(true_is_k, prob_k) if np.unique(true_is_k).size == 2 else np.nan
            if not np.isnan(k_auc):
                per_k_aucs.append(k_auc)
            actual_k_hours = int(true_is_k.sum())
            pred_k_hours = int(pred_is_k.sum())
            stage1_per_count_rows.append({
                "group": model_key,
                "count_k": k,
                "binary_roc_auc": k_auc,
                "binary_accuracy": float((pred_is_k == true_is_k).mean()),
                "recall_when_true_k": float(pred_is_k[true_is_k == 1].mean()) if actual_k_hours > 0 else np.nan,
                "precision_when_pred_k": float(true_is_k[pred_is_k == 1].mean()) if pred_k_hours > 0 else np.nan,
                "actual_rate": float(true_is_k.mean()),
                "predicted_rate": float(pred_is_k.mean()),
                "actual_hours": actual_k_hours,
                "predicted_hours": pred_k_hours,
            })

        signed_count_diff = flat_pred_counts.astype(np.int16) - flat_true_counts.astype(np.int16)
        stage1_summary_rows.append({
            "group": model_key,
            "macro_roc_auc_ovr": float(np.mean(per_k_aucs)) if per_k_aucs else np.nan,
            "exact_count_accuracy": float((flat_pred_counts == flat_true_counts).mean()),
            "within_1_count_accuracy": float((np.abs(signed_count_diff) <= 1).mean()),
            "count_mae": float(np.abs(signed_count_diff).mean()),
            "expected_count_mae": float(np.abs(flat_expected_counts - flat_true_counts).mean()),
            "positive_error_rate_overpred": float((signed_count_diff > 0).mean()),
            "negative_error_rate_underpred": float((signed_count_diff < 0).mean()),
            "mean_signed_count_error": float(signed_count_diff.mean()),
            "observations": len(flat_true_counts),
        })

        # Stage 1 by 24h block in the week
        n_blocks = T // block_hours
        for block_idx in range(n_blocks):
            h_start = block_idx * block_hours
            h_end = min((block_idx + 1) * block_hours, T)
            b_true = true_counts[:, h_start:h_end].ravel().astype(np.int16)
            b_pred = pred_counts[:, h_start:h_end].ravel().astype(np.int16)
            b_exp = expected_counts[:, h_start:h_end].ravel()
            b_diff = b_pred - b_true
            stage1_by_24h_rows.append({
                "group": model_key,
                "week_position": f"{block_idx + 1}. 24h (t={h_start:03d}-{h_end - 1:03d})",
                "exact_count_accuracy": float((b_diff == 0).mean()),
                "total_count_error_rate": float((b_diff != 0).mean()),
                "positive_error_rate_overpred": float((b_diff > 0).mean()),
                "negative_error_rate_underpred": float((b_diff < 0).mean()),
                "count_mae": float(np.abs(b_diff).mean()),
                "expected_count_mae": float(np.abs(b_exp - b_true).mean()),
                "mean_signed_count_error": float(b_diff.mean()),
            })

        # Stage 2 evaluation: Stage-1 predicted inputs vs. True (Oracle) 168h count inputs
        extra_inputs = getattr(model, "_stage2_extra_inputs", None)
        use_proba = getattr(model, "_stage2_use_proba", True)
        stage2_models = getattr(model, "_stage2_models", {})

        count_prob_true = _build_true_count_probability_matrix(target_3d, n_units=n_units)
        X_stage2_pred_input = build_stage2_weekly_features(
            X_wide,
            count_prob_pred,
            stage2_extra_inputs=extra_inputs,
        )
        X_stage2_true_input = build_stage2_weekly_features(
            X_wide,
            count_prob_true,
            stage2_extra_inputs=extra_inputs,
        )

        bit_weights = (1 << np.arange(n_units, dtype=np.uint16)).reshape(1, 1, n_units)
        hourly_bitmasks = (target_3d.astype(np.uint16) * bit_weights).sum(axis=2)

        # Reconstruct generator probabilities under 3 scenarios for ablation:
        # 1) Standard (pred Stage-1 counts + Stage-2 with pred inputs)
        # 2) Stage-1 pred counts + Stage-2 with TRUE 168h count inputs
        # 3) TRUE Stage-1 counts + Stage-2 with TRUE 168h count inputs
        unit_prob_standard = np.repeat(count_prob_pred[:, :, n_units : n_units + 1], n_units, axis=2).astype(np.float32)
        unit_prob_s2_true_in = np.repeat(count_prob_pred[:, :, n_units : n_units + 1], n_units, axis=2).astype(np.float32)
        unit_prob_both_true = np.repeat(count_prob_true[:, :, n_units : n_units + 1], n_units, axis=2).astype(np.float32)

        for k in range(1, n_units):
            s2_clf = stage2_models.get(k)
            u_given_k_pred = _predict_stage2_unit_probabilities_given_k(
                s2_clf, X_stage2_pred_input, n_units=n_units, k=k, use_proba=use_proba
            )
            u_given_k_true = _predict_stage2_unit_probabilities_given_k(
                s2_clf, X_stage2_true_input, n_units=n_units, k=k, use_proba=use_proba
            )

            unit_prob_standard += count_prob_pred[:, :, k : k + 1] * u_given_k_pred[:, np.newaxis, :]
            unit_prob_s2_true_in += count_prob_pred[:, :, k : k + 1] * u_given_k_true[:, np.newaxis, :]
            unit_prob_both_true += count_prob_true[:, :, k : k + 1] * u_given_k_true[:, np.newaxis, :]

            if s2_clf is None:
                continue

            observed_masks = s2_clf._combination_bitmasks
            comb_unit_matrix = s2_clf._combination_unit_matrix
            _, pred_comb_idx_from_pred = _predict_stage2_combination_classes(s2_clf, X_stage2_pred_input, len(observed_masks))
            _, pred_comb_idx_from_true = _predict_stage2_combination_classes(s2_clf, X_stage2_true_input, len(observed_masks))

            pred_masks_from_pred = observed_masks[pred_comb_idx_from_pred]
            pred_masks_from_true = observed_masks[pred_comb_idx_from_true]
            pred_units_hard_pred = comb_unit_matrix[pred_comb_idx_from_pred]  # (n_rows, n_units)
            pred_units_hard_true = comb_unit_matrix[pred_comb_idx_from_true]  # (n_rows, n_units)

            weekly_correct_pred = []
            weekly_correct_true = []
            weekly_hours_k = []
            hourly_unit_matches_pred = []
            hourly_unit_matches_true = []
            hourly_exact_comb_pred = []
            hourly_exact_comb_true = []

            for week_idx in range(n_rows):
                mask_k = true_counts[week_idx] == k
                n_hk = int(mask_k.sum())
                if n_hk == 0:
                    continue
                masks_w, counts_w = np.unique(hourly_bitmasks[week_idx, mask_k], return_counts=True)
                true_dom_mask = int(masks_w[np.argmax(counts_w)])

                weekly_correct_pred.append(int(pred_masks_from_pred[week_idx] == true_dom_mask))
                weekly_correct_true.append(int(pred_masks_from_true[week_idx] == true_dom_mask))
                weekly_hours_k.append(n_hk)

                true_units_in_k = target_3d[week_idx, mask_k, :]  # (n_hk, n_units)
                hourly_unit_matches_pred.append((true_units_in_k == pred_units_hard_pred[week_idx]).mean())
                hourly_unit_matches_true.append((true_units_in_k == pred_units_hard_true[week_idx]).mean())
                hourly_exact_comb_pred.append((hourly_bitmasks[week_idx, mask_k] == pred_masks_from_pred[week_idx]).mean())
                hourly_exact_comb_true.append((hourly_bitmasks[week_idx, mask_k] == pred_masks_from_true[week_idx]).mean())

            if weekly_hours_k:
                weights = np.array(weekly_hours_k, dtype=np.float64)
                stage2_comparison_rows.append({
                    "group": model_key,
                    "count_k": k,
                    "weeks_with_k": len(weekly_hours_k),
                    "hours_with_k": int(weights.sum()),
                    "weekly_comb_acc_pred_input": float(np.mean(weekly_correct_pred)),
                    "weekly_comb_acc_true_input": float(np.mean(weekly_correct_true)),
                    "weighted_comb_acc_pred_input": float(np.average(weekly_correct_pred, weights=weights)),
                    "weighted_comb_acc_true_input": float(np.average(weekly_correct_true, weights=weights)),
                    "hourly_exact_set_acc_pred_input": float(np.average(hourly_exact_comb_pred, weights=weights)),
                    "hourly_exact_set_acc_true_input": float(np.average(hourly_exact_comb_true, weights=weights)),
                    "hourly_unit_acc_at_true_k_pred_input": float(np.average(hourly_unit_matches_pred, weights=weights)),
                    "hourly_unit_acc_at_true_k_true_input": float(np.average(hourly_unit_matches_true, weights=weights)),
                })

        for u_idx, unit in enumerate(units):
            actual_u = target_3d[:, :, u_idx].ravel()
            p_std = unit_prob_standard[:, :, u_idx].ravel()
            p_s2_true = unit_prob_s2_true_in[:, :, u_idx].ravel()
            p_both_true = unit_prob_both_true[:, :, u_idx].ravel()

            has_both = np.unique(actual_u).size == 2
            stage2_ablation_rows.append({
                "generator": unit,
                "roc_auc_standard": roc_auc_score(actual_u, p_std) if has_both else np.nan,
                "roc_auc_stage2_with_true_inputs": roc_auc_score(actual_u, p_s2_true) if has_both else np.nan,
                "roc_auc_oracle_stage1_and_stage2_true_inputs": roc_auc_score(actual_u, p_both_true) if has_both else np.nan,
                "acc_standard": accuracy_score(actual_u, (p_std >= 0.5).astype(np.uint8)),
                "acc_stage2_with_true_inputs": accuracy_score(actual_u, (p_s2_true >= 0.5).astype(np.uint8)),
                "acc_oracle_stage1_and_stage2_true_inputs": accuracy_score(actual_u, (p_both_true >= 0.5).astype(np.uint8)),
            })

    return {
        "stage1_summary": pd.DataFrame(stage1_summary_rows),
        "stage1_per_count": pd.DataFrame(stage1_per_count_rows),
        "stage1_by_24h": pd.DataFrame(stage1_by_24h_rows),
        "stage2_comparison": pd.DataFrame(stage2_comparison_rows),
        "stage2_generator_ablation": pd.DataFrame(stage2_ablation_rows),
    }


def evaluate_errors_by_week_position(y_true: pd.DataFrame, predictions: pd.DataFrame, block_hours: int = 24) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Compute error metrics by 24-hour position in the week (Day 1..7), overall and per generator."""
    units = get_unit_names(y_true.columns)
    n_blocks = T // block_hours

    overall_rows = []
    per_generator_rows = []

    for block_idx in range(n_blocks):
        h_start = block_idx * block_hours
        h_end = min((block_idx + 1) * block_hours, T)
        block_label = f"{block_idx + 1}. 24h (t={h_start:03d}-{h_end - 1:03d})"

        block_cols = [
            f"result_committed_{unit}_t{hour}"
            for unit in units
            for hour in range(h_start, h_end)
        ]
        actual_all = y_true[block_cols].to_numpy(dtype=np.float32).ravel()
        pred_all = predictions[block_cols].to_numpy(dtype=np.float32).ravel()
        pred_label_all = (pred_all >= 0.5).astype(np.float32)
        prob_diff_all = pred_all - actual_all

        fp_all = float(((pred_label_all == 1) & (actual_all == 0)).mean())
        fn_all = float(((pred_label_all == 0) & (actual_all == 1)).mean())
        overall_rows.append({
            "week_position": block_label,
            "roc_auc": roc_auc_score(actual_all, pred_all) if np.unique(actual_all).size == 2 else np.nan,
            "accuracy_at_0.5": float((pred_label_all == actual_all).mean()),
            "total_error_rate": fp_all + fn_all,
            "positive_error_rate_fp": fp_all,
            "negative_error_rate_fn": fn_all,
            "net_signed_error_rate": fp_all - fn_all,
            "mean_pos_prob_error": float(np.maximum(prob_diff_all, 0.0).mean()),
            "mean_neg_prob_error": float(np.minimum(prob_diff_all, 0.0).mean()),
            "mean_signed_prob_error": float(prob_diff_all.mean()),
            "prob_mae": float(np.abs(prob_diff_all).mean()),
        })

        for unit in units:
            u_cols = [f"result_committed_{unit}_t{hour}" for hour in range(h_start, h_end)]
            actual_u = y_true[u_cols].to_numpy(dtype=np.float32).ravel()
            pred_u = predictions[u_cols].to_numpy(dtype=np.float32).ravel()
            pred_label_u = (pred_u >= 0.5).astype(np.float32)
            prob_diff_u = pred_u - actual_u

            fp_u = float(((pred_label_u == 1) & (actual_u == 0)).mean())
            fn_u = float(((pred_label_u == 0) & (actual_u == 1)).mean())
            per_generator_rows.append({
                "generator": unit,
                "week_position": block_label,
                "block_index": block_idx + 1,
                "roc_auc": roc_auc_score(actual_u, pred_u) if np.unique(actual_u).size == 2 else np.nan,
                "accuracy_at_0.5": float((pred_label_u == actual_u).mean()),
                "total_error_rate": fp_u + fn_u,
                "positive_error_rate_fp": fp_u,
                "negative_error_rate_fn": fn_u,
                "net_signed_error_rate": fp_u - fn_u,
                "mean_pos_prob_error": float(np.maximum(prob_diff_u, 0.0).mean()),
                "mean_neg_prob_error": float(np.minimum(prob_diff_u, 0.0).mean()),
                "mean_signed_prob_error": float(prob_diff_u.mean()),
                "prob_mae": float(np.abs(prob_diff_u).mean()),
            })

    return pd.DataFrame(overall_rows), pd.DataFrame(per_generator_rows)


def plot_and_save_timestep_errors(
    y_true: pd.DataFrame,
    predictions: pd.DataFrame,
    fitted_models: dict,
    X_wide: pd.DataFrame,
    output_dir: Path,
    block_hours: int = 24,
) -> pd.DataFrame:
    """Plot and save per-timestep (t=0..167) positive and negative errors for every generator and Stage 1."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    units = get_unit_names(y_true.columns)
    hours = np.arange(T)
    timestep_rows = []

    for unit in units:
        u_cols = [f"result_committed_{unit}_t{hour}" for hour in range(T)]
        actual_mat = y_true[u_cols].to_numpy(dtype=np.float32)  # (n_weeks, T)
        pred_mat = predictions[u_cols].to_numpy(dtype=np.float32)  # (n_weeks, T)
        pred_label_mat = (pred_mat >= 0.5).astype(np.float32)
        prob_diff_mat = pred_mat - actual_mat

        fp_by_t = ((pred_label_mat == 1) & (actual_mat == 0)).mean(axis=0)
        fn_by_t = ((pred_label_mat == 0) & (actual_mat == 1)).mean(axis=0)
        total_err_by_t = fp_by_t + fn_by_t
        net_err_by_t = fp_by_t - fn_by_t

        pos_prob_by_t = np.maximum(prob_diff_mat, 0.0).mean(axis=0)
        neg_prob_by_t = np.minimum(prob_diff_mat, 0.0).mean(axis=0)  # <= 0
        signed_prob_by_t = prob_diff_mat.mean(axis=0)

        for hour in range(T):
            timestep_rows.append({
                "generator": unit,
                "hour": hour,
                "week_day_block": (hour // block_hours) + 1,
                "positive_error_rate_fp": float(fp_by_t[hour]),
                "negative_error_rate_fn": float(fn_by_t[hour]),
                "total_error_rate": float(total_err_by_t[hour]),
                "net_signed_error_rate": float(net_err_by_t[hour]),
                "mean_pos_prob_error": float(pos_prob_by_t[hour]),
                "mean_neg_prob_error": float(neg_prob_by_t[hour]),
                "mean_signed_prob_error": float(signed_prob_by_t[hour]),
            })

        fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)
        ax1, ax2 = axes

        ax1.fill_between(hours, 0, fp_by_t, color="#d95f02", alpha=0.45, label="Positiver Fehler (+FP: Vorhersage=1, Wahr=0)")
        ax1.fill_between(hours, 0, -fn_by_t, color="#1f78b4", alpha=0.45, label="Negativer Fehler (-FN: Vorhersage=0, Wahr=1)")
        ax1.plot(hours, net_err_by_t, color="#222222", linewidth=1.4, label="Netto-Fehlerrate (FP - FN)")
        ax1.plot(hours, total_err_by_t, color="#7570b3", linestyle="--", linewidth=1.1, alpha=0.85, label="Gesamtfehlerrate (FP + FN)")
        ax1.axhline(0.0, color="black", linewidth=0.8)

        for b_idx in range(T // block_hours):
            h0 = b_idx * block_hours
            h1 = min((b_idx + 1) * block_hours, T)
            if h0 > 0:
                ax1.axvline(h0, color="gray", linestyle=":", alpha=0.7)
                ax2.axvline(h0, color="gray", linestyle=":", alpha=0.7)
            b_err = total_err_by_t[h0:h1].mean()
            b_fp = fp_by_t[h0:h1].mean()
            b_fn = fn_by_t[h0:h1].mean()
            ax1.text(
                h0 + block_hours / 2,
                0.93,
                f"{b_idx + 1}. 24h\nErr:{b_err:.2f} (+{b_fp:.2f}/-{b_fn:.2f})",
                transform=ax1.get_xaxis_transform(),
                ha="center",
                va="top",
                fontsize=8,
                bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.75, edgecolor="none"),
            )

        ax1.set_title(f"{unit} – Klassifikationsfehler (@0.5) pro Zeitschritt der Woche (t=0..167)")
        ax1.set_ylabel("Fehlerrate (-FN / +FP)")
        ax1.legend(loc="lower right", fontsize=8, ncol=2)
        ax1.grid(True, alpha=0.25)

        ax2.fill_between(hours, 0, pos_prob_by_t, color="#e6550d", alpha=0.45, label="Positiver Wahrscheinlichkeitsfehler max(p - y, 0)")
        ax2.fill_between(hours, 0, neg_prob_by_t, color="#3182bd", alpha=0.45, label="Negativer Wahrscheinlichkeitsfehler min(p - y, 0)")
        ax2.plot(hours, signed_prob_by_t, color="#111111", linewidth=1.4, label="Mittlerer Bias E[p - y]")
        ax2.axhline(0.0, color="black", linewidth=0.8)
        ax2.set_title(f"{unit} – Kontinuierlicher Wahrscheinlichkeitsfehler (p - y) pro Zeitschritt")
        ax2.set_xlabel("Zeitschritt in der Woche (Stunde t = 0 ... 167)")
        ax2.set_ylabel("Wahrscheinlichkeitsfehler")
        ax2.set_xticks(np.arange(0, T + 1, 24))
        ax2.legend(loc="lower right", fontsize=8, ncol=3)
        ax2.grid(True, alpha=0.25)

        fig.tight_layout()
        safe_unit_name = unit.replace(" ", "_").replace(":", "_")
        fig.savefig(output_dir / f"error_timesteps_{safe_unit_name}.png", dpi=150)
        plt.close(fig)

    # Also plot Stage-1 count errors across timesteps t=0..167 for multi-generator groups
    X_long = wide_to_long_X(X_wide)
    n_rows = len(X_wide)
    for model_key, model in fitted_models.items():
        if JOINT_GROUP_SEPARATOR not in model_key:
            continue
        group_units = model_key.split(JOINT_GROUP_SEPARATOR)
        n_units = len(group_units)
        target_3d = np.stack([
            y_true[[f"result_committed_{u}_t{hour}" for hour in range(T)]].to_numpy(dtype=np.int16)
            for u in group_units
        ], axis=2)
        true_counts = target_3d.sum(axis=2)  # (n_rows, T)
        count_prob_pred = _stage1_count_probability_matrix(model, X_long, n_rows=n_rows, n_units=n_units)
        pred_counts = count_prob_pred.argmax(axis=2).astype(np.int16)
        count_diff = pred_counts - true_counts

        overpred_rate_t = (count_diff > 0).mean(axis=0)
        underpred_rate_t = (count_diff < 0).mean(axis=0)
        total_err_rate_t = (count_diff != 0).mean(axis=0)
        pos_mag_t = np.maximum(count_diff, 0).mean(axis=0)
        neg_mag_t = np.minimum(count_diff, 0).mean(axis=0)
        signed_mag_t = count_diff.mean(axis=0)

        fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)
        ax1, ax2 = axes
        ax1.fill_between(hours, 0, overpred_rate_t, color="#d95f02", alpha=0.45, label="Positiver Fehler (Zu viele Generatoren vorhergesagt)")
        ax1.fill_between(hours, 0, -underpred_rate_t, color="#1f78b4", alpha=0.45, label="Negativer Fehler (Zu wenige Generatoren vorhergesagt)")
        ax1.plot(hours, overpred_rate_t - underpred_rate_t, color="#222222", linewidth=1.4, label="Netto-Richtungsfehler")
        ax1.plot(hours, total_err_rate_t, color="#7570b3", linestyle="--", linewidth=1.1, label="Gesamtfehlerrate (1 - Exact Count Acc)")
        ax1.axhline(0.0, color="black", linewidth=0.8)
        for b_idx in range(T // block_hours):
            h0 = b_idx * block_hours
            h1 = min((b_idx + 1) * block_hours, T)
            if h0 > 0:
                ax1.axvline(h0, color="gray", linestyle=":", alpha=0.7)
                ax2.axvline(h0, color="gray", linestyle=":", alpha=0.7)
            b_acc = 1.0 - total_err_rate_t[h0:h1].mean()
            ax1.text(
                h0 + block_hours / 2,
                0.93,
                f"{b_idx + 1}. 24h\nAcc:{b_acc:.2f}",
                transform=ax1.get_xaxis_transform(),
                ha="center",
                va="top",
                fontsize=8,
                bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.75, edgecolor="none"),
            )
        ax1.set_title(f"Vorstufe (Stage 1) {model_key} – Fehlerrate der Generatoranzahl pro Zeitschritt")
        ax1.set_ylabel("Fehlerrate (-Unter / +Über)")
        ax1.legend(loc="lower right", fontsize=8, ncol=2)
        ax1.grid(True, alpha=0.25)

        ax2.fill_between(hours, 0, pos_mag_t, color="#e6550d", alpha=0.45, label="Mittlere Überschätzung (Anzahl Generatoren)")
        ax2.fill_between(hours, 0, neg_mag_t, color="#3182bd", alpha=0.45, label="Mittlere Unterschätzung (Anzahl Generatoren)")
        ax2.plot(hours, signed_mag_t, color="#111111", linewidth=1.4, label="Mittlere Abweichung E[pred_count - true_count]")
        ax2.axhline(0.0, color="black", linewidth=0.8)
        ax2.set_title(f"Vorstufe (Stage 1) {model_key} – Mittlere Abweichung der Generatoranzahl")
        ax2.set_xlabel("Zeitschritt in der Woche (Stunde t = 0 ... 167)")
        ax2.set_ylabel("Abweichung (Generatoren)")
        ax2.set_xticks(np.arange(0, T + 1, 24))
        ax2.legend(loc="lower right", fontsize=8, ncol=3)
        ax2.grid(True, alpha=0.25)

        fig.tight_layout()
        safe_group_name = model_key.replace(JOINT_GROUP_SEPARATOR, "_").replace(" ", "_")
        fig.savefig(output_dir / f"error_timesteps_stage1_{safe_group_name}.png", dpi=150)
        plt.close(fig)

    timestep_df = pd.DataFrame(timestep_rows)
    timestep_df.to_csv(output_dir / "timestep_errors_by_generator.csv", index=False)
    return timestep_df


def show_training_week_example(X, y, week_start=None, generator=None, rows_to_print=24):
    """Show the model input and target for one training week and generator."""
    if "starttime" not in X:
        raise ValueError("X must contain a 'starttime' column.")
    if len(X) == 0 or len(y) == 0:
        raise ValueError("X and y must contain training data.")
    if not X.index.equals(y.index):
        raise ValueError("X and y must have identical indexes.")

    units = get_unit_names(y.columns)
    if generator is None:
        generator = units[0]
    if generator not in units:
        raise ValueError(f"Unknown generator {generator!r}. Choose one of: {units}")

    available_starts = pd.to_datetime(X["starttime"])
    if week_start is None:
        selected_start = available_starts.iloc[0]
    else:
        requested_start = pd.Timestamp(week_start)
        matching_starts = available_starts[available_starts >= requested_start]
        if matching_starts.empty:
            raise ValueError("No training week starts on or after week_start.")
        selected_start = matching_starts.iloc[0]

    selected_index = available_starts[available_starts == selected_start].index
    X_week = X.loc[selected_index]
    y_week = y.loc[selected_index]
    model_input = wide_to_long_X(X_week)
    target_columns = [f"result_committed_{generator}_t{hour}" for hour in range(T)]
    target_values = y_week[target_columns].to_numpy(dtype=np.uint8).ravel()

    example = model_input.copy()
    example.insert(1, "starttime", selected_start)
    example["generator"] = generator
    example["target_commitment"] = target_values

    print(f"Training-week example: {selected_start} to {selected_start + pd.Timedelta(hours=T - 1)}")
    print(f"Generator: {generator}")
    print(f"Model input shape: {model_input.shape}")
    print(f"Target column: result_committed_{generator}_t0 ... result_committed_{generator}_t167")
    print(f"Showing the first {min(rows_to_print, len(example))} of {len(example)} hourly rows:")
    print(example.head(rows_to_print).to_string(index=False))
    return example


## Data Preprocessing

In [71]:
# Topology Information:

## Generator Locations:
gen_locations = {
    "Byrte": ["Byrte_G1"],
    "Haukeli": ["Haukeli_G1"],
    "Hogga": ["Hogga_G1"],
    "Kjela": ["Kjela_G1"],
    "Lio": ["Lio_G1"],
    "Songa": ["Songa_G1"],
    "Tokke": ["Tokke_G1", "Tokke_G3", "Tokke_G4", "Tokke_G2"],
    "Vesle Kjela": ["Vesle Kjela_G1"],
    "Vinje": ["Vinje_G1", "Vinje_G2", "Vinje_G3"],
}

In [72]:
## Define the training set
class Training_advanced(Training_set):
    
    def raw_preprocessing(self):
        raw = self.load_copies()
        
        # 1. Extract raw dataframes for price, volume, and water value
        price_df = raw["day_ahead_price"].copy()
        volume_df = raw["volume"].copy()
        water_value_df = raw["water_value"].copy()
        inflows = raw["inflow"].copy()

        # Hourly cyclical time features directly in preprocessing
        hour = price_df.index.hour
        price_df["hour"] = hour
        price_df["hour_sin"] = np.sin(2 * np.pi * hour / 24)
        price_df["hour_cos"] = np.cos(2 * np.pi * hour / 24)
        
        # 2. Prices: Daily Relative Price (Normalized by daily mean)
        price_df["date_only"] = price_df.index.normalize()
        daily_mean_price = price_df.groupby("date_only")["day_ahead_price"].transform("mean")
        price_df["price_relative_to_daily_mean"] = price_df["day_ahead_price"] - daily_mean_price
        price_df.drop(columns=["date_only"], inplace=True)

        # 2.2 Add a price rank feature how often the price is higher than the current price in the next 24 hours
        price_df["price_rank_next_24h"] = (
            price_df["day_ahead_price"]
            .shift(-23)
            .rolling(window=24, min_periods=1)
            .apply(lambda x: (x > x[0]).sum(), raw=True)
        )

        # 3. Engineer Feature B: Economic Spreads (Price minus Water Value per reservoir)
        wv_hourly = water_value_df.reindex(price_df.index, method="ffill")

        # 4. For each reservoir compute a spread colum: Price minus Water Value
        spread_frames = {}
        for res_col in water_value_df.columns:
            # Create a spread column for each reservoir: Price minus Water Value
            spread_col_name = f"spread_{res_col}"
            spread_frames[spread_col_name] = price_df["day_ahead_price"] - wv_hourly[res_col]
            
        spread_df = pd.DataFrame(spread_frames, index=price_df.index)

        # 5. Engineer Feature C: Water Value Difference (Forward Difference) - Is the water value increasing or decreasing?
        wv_diff_forward = water_value_df.shift(-1) - water_value_df
        wv_diff_forward.columns = [f"{col}_wv_diff_forward" for col in wv_diff_forward.columns]
        
        wv_hourly = wv_diff_forward.reindex(price_df.index, method="ffill")

        # 5. Day feature day of week
        day_features_df = pd.DataFrame(index=water_value_df.index)
        day_features_df["dayofweek"] = day_features_df.index.dayofweek
        day_features_df["is_weekend"] = (day_features_df.index.dayofweek >= 5).astype(int)


        # 6. Volumetric features: current volume, volume above minimum, and room to maximum.
        topology_path = repo_root / "sintef_project" / "data" / "extended" / "Tokke_Vinje_topology.yaml"
        with topology_path.open(encoding="utf-8") as topology_file:
            topology = yaml.safe_load(topology_file)
        max_volumes = {
            reservoir: float(details["max_vol"])
            for reservoir, details in topology["model"]["reservoir"].items()
        }

        min_volume_hourly = raw["min_volume"]
        min_volume_daily = min_volume_hourly.reindex(volume_df.index)
        min_volume_daily = min_volume_daily.reindex(columns=volume_df.columns).fillna(0.0)
        missing_max_volumes = sorted(set(volume_df.columns) - set(max_volumes))
        if missing_max_volumes:
            raise ValueError(f"Missing max_vol for reservoirs: {missing_max_volumes}")

        # Calculate the percentage of available volume relative to the maximum volume - minimum volume for each reservoir
        max_vol_series = pd.Series(max_volumes).reindex(volume_df.columns)
        available_volume_percentage_df = (volume_df - min_volume_daily) / (max_vol_series - min_volume_daily)

        # 7. Inflows of the lakes and the one given river inflow.
        inflow_df = raw["inflow"].copy()
        future_inflow_frames = {}
        
        for col in inflow_df.columns:
            # Calculate rolling sum looking forward (next 24 hours)
            future_inflow_frames[f"{col}_inflow_sum_next_24h"] = (
                inflow_df[col].iloc[::-1].rolling(window=24, min_periods=1).sum().iloc[::-1]
            )
            # Calculate rolling sum looking forward (next 72 hours)
            future_inflow_frames[f"{col}_inflow_sum_next_72h"] = (
                inflow_df[col].iloc[::-1].rolling(window=72, min_periods=1).sum().iloc[::-1]
            )
            
        future_inflow_df = pd.DataFrame(future_inflow_frames, index=inflow_df.index)


        # Combine everything back into the dictionary that prepare_data expects
        processed_sources = {
            "volume": volume_df,
            "inflow": inflows,
            "day_features": day_features_df,
            "price": price_df,
            "spread": spread_df,
            "water_value": wv_diff_forward,
            "available_volume": available_volume_percentage_df,
            "future_inflow": future_inflow_df,
        }

        return processed_sources

training_set = Training_advanced()

In [73]:
## Defining the training and test periods
start_date = datetime(2015, 1, 1)
end_date = datetime(2022, 12, 31)

hour_memory = 72

X, y = training_set.prepare_data(
    start_date=start_date,
    end_date=end_date,
    hour_memory=hour_memory,
)

training_week_example = show_training_week_example(
    X,
    y,
    week_start="2015-01-02",
    generator=get_unit_names(y.columns)[0],
)

# Save as a csv to inspect the training week example
training_week_example.to_csv(repo_root / "output" / "training_week_example_advanced.csv", index=False)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("X index:", X.index)
print("y index:", y.index)

print("Same index:", X.index.equals(y.index))
print("Same number of rows:", len(X) == len(y))

# Split the data into training, validation, and test sets based on the starttime column
train_val_mask = X['starttime'] < datetime(2022, 1, 1)
test_mask = X['starttime'] >= datetime(2022, 1, 1)

X_train_val = X.loc[train_val_mask].copy()
y_train_val = y.loc[train_val_mask].copy()

training_set.export_X_and_Y_summary(X_train_val, y_train_val, name="base_model_train_val")

X_test = X.loc[test_mask].copy()
y_test = y.loc[test_mask].copy()


folds = [
    {
        'train_end': '2018-01-01',
        'val_start': '2018-01-01',
        'val_end': '2019-01-01'
    },
    {
        'train_end': '2019-01-01',
        'val_start': '2019-01-01',
        'val_end': '2020-01-01'
    },
    {
        'train_end': '2020-01-01',
        'val_start': '2020-01-01',
        'val_end': '2021-01-01'
    },
    {
        'train_end': '2021-01-01',
        'val_start': '2021-01-01',
        'val_end': '2022-01-01'
    }
]

train_folds = []

for fold in folds:

    train_mask = (
        X['starttime'] < pd.Timestamp(fold['train_end'])
    )

    val_mask = (
        (X['starttime'] >= pd.Timestamp(fold['val_start'])) &
        (X['starttime'] < pd.Timestamp(fold['val_end']))
    )

    X_train = X.loc[train_mask]
    y_train = y.loc[train_mask]

    X_val = X.loc[val_mask]
    y_val = y.loc[val_mask]

    train_folds.append((X_train, y_train, X_val, y_val))

    print(
        f"Train: {X_train['starttime'].min()} "
        f"to {X_train['starttime'].max()}"
    )

    print(
        f"Validation: {X_val['starttime'].min()} "
        f"to {X_val['starttime'].max()}"
    )

Training-week example: 2015-01-04 00:00:00 to 2015-01-10 23:00:00
Generator: Byrte_G1
Model input shape: (168, 132)
Target column: result_committed_Byrte_G1_t0 ... result_committed_Byrte_G1_t167
Showing the first 24 of 168 hourly rows:
     hour_sin  starttime      hour_cos  future_inflow_Bandak_inflow_sum_next_24h  future_inflow_Bandak_inflow_sum_next_72h  future_inflow_Bitdalsvatn_inflow_sum_next_24h  future_inflow_Bitdalsvatn_inflow_sum_next_72h  future_inflow_Bordalsvatn_inflow_sum_next_24h  future_inflow_Bordalsvatn_inflow_sum_next_72h  future_inflow_Botnedalsvatn_inflow_sum_next_24h  future_inflow_Botnedalsvatn_inflow_sum_next_72h  future_inflow_Byrtevatn_inflow_sum_next_24h  future_inflow_Byrtevatn_inflow_sum_next_72h  future_inflow_Foersvatn_inflow_sum_next_24h  future_inflow_Foersvatn_inflow_sum_next_72h  future_inflow_Hyljelihyl_inflow_sum_next_24h  future_inflow_Hyljelihyl_inflow_sum_next_72h  future_inflow_Kjelavatn_inflow_sum_next_24h  future_inflow_Kjelavatn_inflow_sum_ne

In [74]:

# Define the base classifier (n_jobs=-1 is crucial here to use all CPU cores!)
estimator = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    n_jobs=-1,  
    random_state=42,
)

# Stage-2 weekly classifier for multi-generator plants (one tree model per active count k in {1, ..., M-1})
stage2_estimator = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    n_estimators=100,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    n_jobs=-1,
    random_state=42,
)

# Optional additional weekly inputs from X_wide for Stage-2 classification.
# Set to None to use only the Stage-1 forecast across the 168 future hours,
# or provide column names/prefixes from X_wide, e.g.:
# stage2_extra_inputs = ["available_volume_", "volume_", "water_value_", "start_month_"]
stage2_extra_inputs = None

# Optional time-based cross-validation
avg_score = None
bool_train_folds = True
if bool_train_folds:
    validation_scores = []
    for i, (X_train_full, y_train, X_val_full, y_val) in enumerate(train_folds):
        print(f"\n--- Starting Training for Fold {i+1} ---")
        time_start = datetime.now()
        print(f"Training period: {X_train_full['starttime'].min()} to {X_train_full['starttime'].max()}")
        print(f"Validation period: {X_val_full['starttime'].min()} to {X_val_full['starttime'].max()}")
        
        # Fit one long-format model per generator (and two-stage models for multi-generator plants)
        model = fit_long_models_with_progress(
            estimator=estimator,
            X_train_wide=X_train_full,
            y_train_wide=y_train,
            joint_groups=get_joint_generator_groups(gen_locations),
            stage2_estimator=stage2_estimator,
            stage2_extra_inputs=stage2_extra_inputs,
        )
        
        # Generate predictions
        val_predictions = positive_class_probabilities(model, X_val_full, y_train.columns)
        
        # Calculate and store the score
        fold_score = score(y_val, val_predictions)
        validation_scores.append(fold_score)
        print(f"-> Fold {i+1} Validation micro ROC-AUC: {fold_score:.4f}")
        print(f"-> Fold {i+1} Training Time: {datetime.now() - time_start}")
        
    # Output the average performance across all folds
    avg_score = sum(validation_scores) / len(validation_scores)
    print(f"\n=========================================")
    print(f"Average CV ROC-AUC: {avg_score:.4f}")
    print(f"=========================================\n")


# Final training and testing on the held-out 2022 data
print("--- Final Model on 2022 Test Data ---")
test_mask = X['starttime'] >= pd.Timestamp("2022-01-01")
train_val_mask = X['starttime'] < pd.Timestamp("2022-01-01")

# Use all knowledge from 2015-2021 for the final model
y_train_val = y.loc[train_val_mask]

y_test = y.loc[test_mask]

final_model = fit_long_models_with_progress(
    estimator=estimator,
    X_train_wide=X.loc[train_val_mask],
    y_train_wide=y_train_val,
    joint_groups=get_joint_generator_groups(gen_locations),
    stage2_estimator=stage2_estimator,
    stage2_extra_inputs=stage2_extra_inputs,
)

print("Most important features per generator (by gain):")
for generator, model in final_model.items():
    if hasattr(model, "get_booster"):
        scores = model.get_booster().get_score(importance_type='gain')
        sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        
        print(f"\nGenerator: {generator}")
        for feature, gain_value in sorted_scores[:len(sorted_scores)]:
            print(f"  - {feature}: {gain_value:.2f}")

    for k, stage2_clf in getattr(model, "_stage2_models", {}).items():
        comb_labels = getattr(stage2_clf, "_combination_labels", [])
        print(f"\n  [Stage 2 | {generator} | k={k} active] Weekly combinations: {comb_labels}")
        if hasattr(stage2_clf, "get_booster"):
            s2_scores = stage2_clf.get_booster().get_score(importance_type='gain')
            s2_sorted = sorted(s2_scores.items(), key=lambda x: x[1], reverse=True)
            for feature, gain_value in s2_sorted[:10]:
                print(f"    * {feature}: {gain_value:.2f}")

test_predictions = positive_class_probabilities(final_model, X.loc[test_mask], y_train_val.columns,
    )
test_score = score(y_test, test_predictions)
print(f"-> Final Test micro ROC-AUC (2022): {test_score:.4f}")

generator_performance = evaluate_generator_performance(y_test, test_predictions)
print("\n2022 performance per generator:")
print(generator_performance.to_string(index=False, float_format=lambda value: f"{value:.4f}"))

two_stage_eval = evaluate_two_stage_details(final_model, X.loc[test_mask], y_test)
overall_24h_errors, per_generator_24h_errors = evaluate_errors_by_week_position(y_test, test_predictions)

print("\n--- Vorstufe (Stage 1: Anzahl aktiver Generatoren) 2022 Performance ---")
print(two_stage_eval["stage1_summary"].to_string(index=False, float_format=lambda value: f"{value:.4f}"))

print("\n--- Vorstufe (Stage 1) je Generatoranzahl k ---")
print(two_stage_eval["stage1_per_count"].to_string(index=False, float_format=lambda value: f"{value:.4f}"))

print("\n--- Vorstufe (Stage 1) nach Position in der Woche (24h-Blöcke) ---")
print(two_stage_eval["stage1_by_24h"].to_string(index=False, float_format=lambda value: f"{value:.4f}"))

print("\n--- Zweites Modell (Stage 2): Vorhergesagte Inputs vs. Wahre (Oracle) 168h-Inputs ---")
print(two_stage_eval["stage2_comparison"].to_string(index=False, float_format=lambda value: f"{value:.4f}"))

print("\n--- End-to-End Generator-Vergleich (Standard vs. Stage 2 mit wahren Inputs vs. Wahre Stage 1 + Stage 2) ---")
print(two_stage_eval["stage2_generator_ablation"].to_string(index=False, float_format=lambda value: f"{value:.4f}"))

print("\n--- Fehler nach Position in der Woche (1. bis 7. 24h-Block, Gesamt) ---")
print(overall_24h_errors.to_string(index=False, float_format=lambda value: f"{value:.4f}"))

# Save all plots, CSV tables, and tracking logs in the same folder
tracking_output_dir = repo_root / "output" / "adv_model_error_tracking"
timestep_error_df = plot_and_save_timestep_errors(
    y_true=y_test,
    predictions=test_predictions,
    fitted_models=final_model,
    X_wide=X.loc[test_mask],
    output_dir=tracking_output_dir,
)
generator_performance.to_csv(tracking_output_dir / "generator_performance_2022.csv", index=False)
overall_24h_errors.to_csv(tracking_output_dir / "errors_by_24h_block_overall.csv", index=False)
per_generator_24h_errors.to_csv(tracking_output_dir / "errors_by_24h_block_per_generator.csv", index=False)
for table_name, table_df in two_stage_eval.items():
    table_df.to_csv(tracking_output_dir / f"{table_name}_2022.csv", index=False)

print(f"\nZeitschritt-Fehlerplots und Tabellen gespeichert in: {tracking_output_dir}")

### Add the average ROC-AUC across folds and the final test score to a log file to keep track of model performance over time
additional_context = (
    "Two-stage XGBoost model for multi-generator plants (Stage 1 hourly count + Stage 2 weekly combination trees). "
    "Engineered features: daily relative price, economic spreads, water value forward differences, cyclical time features, "
    "relative available volume, and future inflow sums (hour_memory=72)."
)

log_Section_lines = [
    f"Date: {datetime.now()}",
    f"Context: {additional_context}",
    f"Tracking & Plots Directory: {tracking_output_dir}",
]
if avg_score is not None:
    log_Section_lines.append(f"Average CV ROC-AUC: {avg_score:.4f}")
log_Section_lines.append(f"Final Test ROC-AUC (2022): {test_score:.4f}")
log_Section_lines.append("\nPer-generator 2022 performance:")
for _, row in generator_performance.iterrows():
    log_Section_lines.append(
        f"{row['generator']}: ROC-AUC={row['roc_auc']:.4f}, "
        f"accuracy@0.5={row['accuracy_at_0.5']:.4f}, "
        f"actual_rate={row['actual_commitment_rate']:.4f}, "
        f"predicted_rate@0.5={row['predicted_commitment_rate_at_0.5']:.4f}, "
        f"observations={int(row['observations'])}"
    )

log_Section_lines.append("\nStage 1 (Vorstufe - Anzahl aktiver Generatoren) Summary:")
log_Section_lines.append(two_stage_eval["stage1_summary"].to_string(index=False, float_format=lambda v: f"{v:.4f}"))

log_Section_lines.append("\nStage 1 (Vorstufe) per Count k:")
log_Section_lines.append(two_stage_eval["stage1_per_count"].to_string(index=False, float_format=lambda v: f"{v:.4f}"))

log_Section_lines.append("\nStage 1 (Vorstufe) by 24h Week Position:")
log_Section_lines.append(two_stage_eval["stage1_by_24h"].to_string(index=False, float_format=lambda v: f"{v:.4f}"))

log_Section_lines.append("\nStage 2 (Zweites Modell - Vorhergesagte Inputs vs. Wahre 168h-Inputs):")
log_Section_lines.append(two_stage_eval["stage2_comparison"].to_string(index=False, float_format=lambda v: f"{v:.4f}"))

log_Section_lines.append("\nStage 2 End-to-End Generator Ablation:")
log_Section_lines.append(two_stage_eval["stage2_generator_ablation"].to_string(index=False, float_format=lambda v: f"{v:.4f}"))

log_Section_lines.append("\nOverall Errors by 24h Position in Week (1. 24h to 7. 24h):")
log_Section_lines.append(overall_24h_errors.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

log_Section_lines.append("\nPer-Generator Errors by 24h Position in Week:")
log_Section_lines.append(per_generator_24h_errors.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
log_Section_lines.append("-" * 80 + "\n")
log_entry_text = "\n".join(log_Section_lines)

for target_log_path in [repo_root / "adv_model_performance_log.txt", tracking_output_dir / "adv_model_performance_log.txt"]:
    with open(target_log_path, "a", encoding="utf-8") as log_file:
        log_file.write(log_entry_text)


--- Starting Training for Fold 1 ---
Training period: 2015-01-04 00:00:00 to 2017-12-31 00:00:00
Validation period: 2018-01-01 00:00:00 to 2018-12-31 00:00:00
[1/9] Tokke_G1::Tokke_G3::Tokke_G4::Tokke_G2 (Stage-2 weekly trees per count k: {1: 3, 2: 3, 3: 2}) finished | last model: 0.47 min | elapsed: 0.47 min | ETA: 3.78 min
[2/9] Vinje_G1::Vinje_G2::Vinje_G3 (Stage-2 weekly trees per count k: {1: 3, 2: 1}) finished | last model: 0.31 min | elapsed: 0.79 min | ETA: 2.76 min
[3/9] Byrte_G1 finished | last model: 0.07 min | elapsed: 0.86 min | ETA: 1.72 min
[4/9] Haukeli_G1 finished | last model: 0.07 min | elapsed: 0.93 min | ETA: 1.16 min
[5/9] Hogga_G1 finished | last model: 0.07 min | elapsed: 1.00 min | ETA: 0.80 min
[6/9] Kjela_G1 finished | last model: 0.08 min | elapsed: 1.08 min | ETA: 0.54 min
[7/9] Lio_G1 finished | last model: 0.07 min | elapsed: 1.15 min | ETA: 0.33 min
[8/9] Songa_G1 finished | last model: 0.08 min | elapsed: 1.23 min | ETA: 0.15 min
[9/9] Vesle Kjela_G1 f

## Creating the submission format

In [75]:
## Create Kaggle Submission data
X_kaggle, _ = training_set.prepare_data(
    start_date=datetime(2023, 1, 1),
    end_date=datetime(2024, 12, 25),
    hour_memory=hour_memory,
    prediction_mapping_path=repo_root / "sintef_project" / "prediction_mapping.csv",
)

training_set.export_X_and_Y_summary(X_kaggle, None, name="submission_input_adv_model")

print(
    "Kaggle date range:",
    X_kaggle["starttime"].min(),
    "to",
    X_kaggle["starttime"].max(),
)

kaggle_predictions = positive_class_probabilities(
    fitted_models=final_model,
    X_wide=X_kaggle,
    y_columns=y_train_val.columns,
)


kaggle_predictions.insert(
    0,
    "Run No",
    X_kaggle["Run No"].to_numpy(),
)



# Create Kaggle submission file

submission_path = create_submission(
    predictions=kaggle_predictions,
    identifier="basic_model_without_2022",
)

print(f"Kaggle submissions saved here: {submission_path}")

Kaggle date range: 2023-01-01 00:00:00 to 2024-12-25 00:00:00
Kaggle submissions saved here: output\basic_model_without_2022.csv
